In [1]:
from pathlib import Path
import re
import subprocess
import sys

import eurostat
import numpy as np
import pandas as pd

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists() and (candidate / "README.md").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from src.config import PROJECT_ROOT, ensure_default_directories

ensure_default_directories()

COUNTRY_DIR = PROJECT_ROOT / "6 Country level variables"
OUTPUTS_DIR = COUNTRY_DIR / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"

for folder in [OUTPUTS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

COUNTRY_PANEL_CSV_PATH = TABLES_DIR / "demography_country_year.csv"
COUNTRY_PANEL_FALLBACK_CSV_PATH = TABLES_DIR / "demography_country_year_latest.csv"

assert PROJECT_ROOT.exists(), f"Project root does not exist: {PROJECT_ROOT}"

print(f"Project root: {PROJECT_ROOT}")
print(f"Tables dir:   {TABLES_DIR}")
print(f"Output file:  {COUNTRY_PANEL_CSV_PATH}")

Project root: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics
Tables dir:   C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\6 Country level variables\outputs\tables
Output file:  C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\6 Country level variables\outputs\tables\demography_country_year.csv


# Country-Level Variables (Eurostat)

This notebook builds a country-year demographic panel for downstream merges with MEP-level data.

Run from top to bottom. Paths are configured once in the next cell for portability and easy GitHub reuse.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


# Country-Year Demographic Panel from Eurostat

This notebook builds a clean country-year demographic dataset from Eurostat API data.

## Objective

Create a panel with one row per country-year, suitable for later merge with MEP-year topic-modelling outputs.

## Variables targeted

1. Total fertility rate
2. Old-age dependency ratio
3. Median age
4. Life expectancy
5. Net migration rate
6. Population growth rate
7. GDP per capita index (EU27 = 100)
8. Employment rate (age 55-64)
9. Youth unemployment
10. Debt-to-GDP ratio

## Workflow overview

1. Install/import packages
2. Define variable-to-dataset mapping
3. Download data via Eurostat API
4. Reshape to long format (`country`, `year`, `value`)
5. Keep EU countries only
6. Standardise country and year
7. Merge into one panel
8. Run quality checks
9. Save `demography_country_year.csv`

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [3]:

# Install eurostat if needed, then import dependencies used in this workflow.
try:
    import eurostat
except ImportError:

    subprocess.check_call([sys.executable, "-m", "pip", "install", "eurostat"] )


pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

print("eurostat package version:", getattr(eurostat, "__version__", "unknown"))


eurostat package version: unknown


## Variable Mapping to Eurostat Datasets

Each target variable is mapped to a Eurostat dataset code and one or more preferred filter combinations.

Why multiple filter combinations?
- Eurostat dataset dimensions can vary over time.
- Some datasets may use slightly different category values.
- The code will try combinations in order and keep the first successful non-empty extraction.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [4]:

# Mapping from thesis variable names to Eurostat dataset codes and preferred filters.
# Note: filter keys are applied only if the key exists in the downloaded dataframe.
VARIABLE_SPECS = {
    "fertility_rate": {
        "dataset_code": "demo_find",
        "preferred_filters": [
            {"indic_de": "TOTFERRT"},
            {}
        ]
    },
    "old_age_dependency_ratio": {
        "dataset_code": "demo_pjanind",
        "preferred_filters": [
            {"indic_de": "PC_OLD", "unit": "PC"},
            {"indic_de": "PC_OLD"},
            {}
        ]
    },
    "median_age": {
        "dataset_code": "demo_pjanind",
        "preferred_filters": [
            {"indic_de": "MEDAGEPOP", "unit": "YR"},
            {"indic_de": "MEDAGEPOP"},
            {}
        ]
    },
    "life_expectancy": {
        "dataset_code": "demo_mlexpec",
        "preferred_filters": [
            {"sex": "T", "age": "Y1", "unit": "YR"},
            {"sex": "T", "age": "Y1"},
            {"sex": "T", "unit": "YR"},
            {"sex": "T"},
            {}
        ]
    },
    "net_migration": {
        "dataset_code": "tps00019",
        "preferred_filters": [
            {"indic_de": "CNMIGRATRT"},
            {}
        ]
    },
    "population_change": {
        "dataset_code": "tps00019",
        "preferred_filters": [
            {"indic_de": "GROWRT"},
            {}
        ]
    },
    "gdp_per_capita_index_eu27_100": {
        "dataset_code": "nama_10_pc",
        "preferred_filters": [
            {"na_item": "B1GQ", "unit": "CLV_I20_HAB"},
            {"na_item": "B1GQ", "unit": "CLV_I15_HAB"},
            {"na_item": "B1GQ", "unit": "CLV_I10_HAB"},
            {}
        ]
    },
    "employment_rate_55_64": {
        "dataset_code": "tesem050",
        "preferred_filters": [
            {"sex": "T", "unit": "PC"},
            {"sex": "T"},
            {}
        ]
    },
    "youth_unemployment": {
        "dataset_code": "une_rt_a",
        "preferred_filters": [
            {"age": "Y15-24", "sex": "T", "unit": "PC_ACT"},
            {"age": "Y15-24", "sex": "T"},
            {"age": "Y15-24"},
            {}
        ]
    },
    "debt_to_gdp": {
        "dataset_code": "teina225",
        "preferred_filters": [
            {"sector": "S13", "unit": "PC_GDP", "na_item": "GD"},
            {"sector": "S13", "unit": "PC_GDP"},
            {"unit": "PC_GDP"},
            {}
        ]
    },
    "pension_expenditure_gdp": {
        "dataset_code": "eurotps00103",
        "preferred_filters": [
            {"unit": "PC_GDP"},
            {"unit": "PC"},
            {}
        ]
    },
    "age_started_receiving_old_age_pension": {
        "dataset_code": "lfso_23pens03",
        "preferred_filters": [
            {"sex": "T", "unit": "YR"},
            {"sex": "T"},
            {}
        ]
    },
    "continued_or_stopped_working_at_pension_start": {
        "dataset_code": "lfso_23pens06",
        "preferred_filters": [
            {"sex": "T"},
            {}
        ]
    },
    "labour_force_participation_55_64": {
        "dataset_code": "lfsa_argan",
        "preferred_filters": [
            {"age": "Y55-64", "sex": "T", "unit": "PC"},
            {"age": "Y55-64", "sex": "T"},
            {"age": "Y60-64", "sex": "T", "unit": "PC"},
            {"age": "Y60-64", "sex": "T"},
            {"sex": "T", "unit": "PC"},
            {"sex": "T"},
            {}
        ]
    },
    "old_age_poverty_rate": {
        "dataset_code": "ilc_li02",
        "preferred_filters": [
            {"sex": "T", "unit": "PC"},
            {"sex": "T"},
            {}
        ]
    },
    "projected_old_age_dependency_ratio_2050": {
        "dataset_code": "proj_19ndbi",
        "preferred_filters": [
            {"proj_type": "BASELINE", "time_period": "2050"},
            {"proj_type": "BASELINE", "time_period": "2070"},
            {"proj_type": "BASELINE"},
            {}
        ]
    },
    "foreign_born_population_share": {
        "dataset_code": "migr_pop3ctb",
        "preferred_filters": [
            {"citizen": "TOTAL", "unit": "PC"},
            {"citizen": "TOTAL"},
            {"unit": "PC"},
            {}
        ]
    },
}

VARIABLE_SPECS


{'fertility_rate': {'dataset_code': 'demo_find',
  'preferred_filters': [{'indic_de': 'TOTFERRT'}, {}]},
 'old_age_dependency_ratio': {'dataset_code': 'demo_pjanind',
  'preferred_filters': [{'indic_de': 'PC_OLD', 'unit': 'PC'},
   {'indic_de': 'PC_OLD'},
   {}]},
 'median_age': {'dataset_code': 'demo_pjanind',
  'preferred_filters': [{'indic_de': 'MEDAGEPOP', 'unit': 'YR'},
   {'indic_de': 'MEDAGEPOP'},
   {}]},
 'life_expectancy': {'dataset_code': 'demo_mlexpec',
  'preferred_filters': [{'sex': 'T', 'age': 'Y1', 'unit': 'YR'},
   {'sex': 'T', 'age': 'Y1'},
   {'sex': 'T', 'unit': 'YR'},
   {'sex': 'T'},
   {}]},
 'net_migration': {'dataset_code': 'tps00019',
  'preferred_filters': [{'indic_de': 'CNMIGRATRT'}, {}]},
 'population_change': {'dataset_code': 'tps00019',
  'preferred_filters': [{'indic_de': 'GROWRT'}, {}]},
 'gdp_per_capita_index_eu27_100': {'dataset_code': 'nama_10_pc',
  'preferred_filters': [{'na_item': 'B1GQ', 'unit': 'CLV_I20_HAB'},
   {'na_item': 'B1GQ', 'unit': 'CLV

In [5]:

# EU list (EU-27) as of the period used in the project, plus UK/GB for UK coverage.
EU_COUNTRIES = [
    "AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "EL", "HU",
    "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE",
    "GB", "UK"
]

def find_time_columns(df: pd.DataFrame) -> list[str]:
    """Return columns that look like year/time columns in Eurostat outputs."""
    time_pattern = re.compile(r"^\d{4}(?:[-_].*)?$")
    return [c for c in df.columns if isinstance(c, str) and time_pattern.match(c)]

def standardize_year(value) -> float:
    """Extract YYYY from labels like '2020', '2020-01', etc."""
    if pd.isna(value):
        return np.nan
    match = re.search(r"(\d{4})", str(value))
    return float(match.group(1)) if match else np.nan

def apply_filters_safely(df: pd.DataFrame, filter_dict: dict) -> pd.DataFrame:
    """
    Apply one filter candidate robustly.
    Rules:
    1) If a requested filter column is missing, this candidate is considered invalid.
    2) Matching is case-insensitive on string values.
    3) If requested value is absent, this candidate is invalid (returns empty).
    """
    filtered = df.copy()
    for col, val in filter_dict.items():
        if col not in filtered.columns:
            return filtered.iloc[0:0].copy()

        series_norm = filtered[col].astype(str).str.strip().str.upper()
        val_norm = str(val).strip().upper()

        if val_norm not in set(series_norm.dropna()):
            return filtered.iloc[0:0].copy()

        filtered = filtered[series_norm == val_norm]

    return filtered

def to_country_year_long(df_raw: pd.DataFrame, variable_name: str, filter_candidates: list[dict]) -> tuple[pd.DataFrame, dict]:
    """
    Convert one Eurostat dataset from wide format into standard long format.
    Tries filter candidates in order and keeps the first non-empty result.
    """
    time_cols = find_time_columns(df_raw)
    if not time_cols:
        raise ValueError(f"No time columns found for variable '{variable_name}'.")

    id_cols = [c for c in df_raw.columns if c not in time_cols]
    selected = None
    selected_filter = None

    for candidate in filter_candidates:
        tmp = apply_filters_safely(df_raw, candidate)
        if not tmp.empty:
            selected = tmp
            selected_filter = candidate
            break

    if selected is None:
        selected = df_raw.copy()
        selected_filter = {}

    long_df = selected.melt(
        id_vars=id_cols,
        value_vars=time_cols,
        var_name="year_raw",
        value_name=variable_name,
    )

    # Standardise country column from Eurostat geography naming conventions.
    geo_candidates = ["geo", "geo\\time", "geo\\TIME_PERIOD", "geotime", "country"]
    geo_col = next((c for c in geo_candidates if c in long_df.columns), None)
    if geo_col is None:
        raise ValueError(f"Could not identify country code column for '{variable_name}'.")

    long_df = long_df.rename(columns={geo_col: "country"})
    long_df["country"] = long_df["country"].astype(str).str.upper().str.strip()
    long_df["country"] = long_df["country"].replace({"UK": "GB"})
    long_df["year"] = long_df["year_raw"].apply(standardize_year).astype("Int64")
    long_df[variable_name] = pd.to_numeric(long_df[variable_name], errors="coerce")

    # Keep only country-year-variable and drop rows with invalid keys.
    long_df = long_df[["country", "year", variable_name]].dropna(subset=["country", "year"]).copy()
    long_df["year"] = long_df["year"].astype(int)

    # Restrict to EU countries only.
    long_df = long_df[long_df["country"].isin(EU_COUNTRIES)].copy()

    # Remove duplicate rows that can occur due to extra dimensions not fully filtered.
    long_df = (
        long_df
        .groupby(["country", "year"], as_index=False)[variable_name]
        .mean()
    )

    meta = {
        "selected_filter": selected_filter,
        "rows": int(len(long_df)),
    }
    return long_df, meta


In [ ]:

# Download each variable dataset, convert to long country-year format, and store metadata.
variable_frames = []
download_log = []

for variable_name, spec in VARIABLE_SPECS.items():
    code = spec["dataset_code"]
    preferred_filters = spec.get("preferred_filters", [{}])

    try:
        df_raw = eurostat.get_data_df(code, flags=False)
        if df_raw is None or df_raw.empty:
            raise ValueError("Dataset returned empty dataframe")

        long_df, meta = to_country_year_long(
            df_raw=df_raw,
            variable_name=variable_name,
            filter_candidates=preferred_filters,
        )

        variable_frames.append(long_df)
        download_log.append({
            "variable": variable_name,
            "dataset_code": code,
            "status": "ok",
            "rows_after_transform": meta["rows"],
            "selected_filter": str(meta["selected_filter"]),
        })

    except Exception as e:
        download_log.append({
            "variable": variable_name,
            "dataset_code": code,
            "status": f"failed: {e}",
            "rows_after_transform": 0,
            "selected_filter": "{}",
        })

download_log_df = pd.DataFrame(download_log)
display(download_log_df)

# Merge all variable tables into one country-year panel (outer join keeps maximum coverage).
if not variable_frames:
    raise RuntimeError("No variable datasets were successfully transformed.")

country_year_panel = variable_frames[0].copy()
for df_var in variable_frames[1:]:
    country_year_panel = country_year_panel.merge(df_var, on=["country", "year"], how="outer")

# Standard sort for panel readability and deterministic output.
country_year_panel = country_year_panel.sort_values(["country", "year"]).reset_index(drop=True)

print("Merged panel shape:", country_year_panel.shape)
country_year_panel.head(10)


In [ ]:

# Basic panel diagnostics required before merge to downstream MEP-year data.
missing_summary = country_year_panel.isna().sum().to_frame(name="n_missing")
missing_summary["pct_missing"] = (100 * missing_summary["n_missing"] / len(country_year_panel)).round(2)

duplicate_count = country_year_panel.duplicated(subset=["country", "year"]).sum()
countries_included = sorted(country_year_panel["country"].dropna().unique().tolist())
years_included = sorted(country_year_panel["year"].dropna().astype(int).unique().tolist())

print("Final dataset shape:", country_year_panel.shape)
print("Duplicate country-year rows:", int(duplicate_count))
print("Number of countries:", len(countries_included))
print("Year range:", (min(years_included), max(years_included)) if years_included else "no years")

print("\nFirst 10 rows:")
display(country_year_panel.head(10))

print("\nMissing values summary:")
display(missing_summary.sort_values("n_missing", ascending=False))

print("\nCountries included:")
print(countries_included)

print("\nYears included:")
print(years_included)

sanity_cols = [
    c for c in ["fertility_rate", "life_expectancy", "median_age", "old_age_dependency_ratio", "debt_to_gdp"]
    if c in country_year_panel.columns
]
if sanity_cols:
    sanity = country_year_panel[sanity_cols].agg(["min", "median", "max"]).T
    print("\nSanity check (min/median/max):")
    display(sanity)

# Save output to structured tables folder.
try:
    country_year_panel.to_csv(COUNTRY_PANEL_CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"\nSaved: {COUNTRY_PANEL_CSV_PATH}")
except PermissionError:
    country_year_panel.to_csv(COUNTRY_PANEL_FALLBACK_CSV_PATH, index=False, encoding="utf-8-sig")
    print("\nPrimary file is locked (likely open in Excel).")
    print(f"Saved fallback file instead: {COUNTRY_PANEL_FALLBACK_CSV_PATH}")


Final dataset shape: (1848, 12)
Duplicate country-year rows: 0
Number of countries: 28
Year range: (1960, 2025)

First 10 rows:


,country,year,fertility_rate,old_age_dependency_ratio,median_age,life_expectancy,net_migration,population_change,gdp_per_capita_index_eu27_100,employment_rate_55_64,youth_unemployment,debt_to_gdp
0,AT,1960,2.69,23.063043,35.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AT,1961,2.78,23.084783,35.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AT,1962,2.80,23.176087,35.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AT,1963,2.82,23.297826,35.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AT,1964,2.79,23.441304,35.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,AT,1965,2.70,23.619565,35.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,AT,1966,2.66,23.854348,34.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,AT,1967,2.62,24.039130,34.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,AT,1968,2.58,24.115217,34.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,AT,1969,2.49,24.243478,34.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Missing values summary:


,n_missing,pct_missing
debt_to_gdp,1524,82.47
population_change,1517,82.09
net_migration,1517,82.09
employment_rate_55_64,1381,74.73
youth_unemployment,1380,74.68
gdp_per_capita_index_eu27_100,975,52.76
life_expectancy,501,27.11
fertility_rate,349,18.89
median_age,209,11.31
old_age_dependency_ratio,154,8.33



Countries included:
['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'GB', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK']

Years included:
[1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Sanity check (min/median/max):


,min,median,max
fertility_rate,1.010000,1.660000,4.07
life_expectancy,66.600000,75.600000,83.30
median_age,26.400000,36.400000,49.10
old_age_dependency_ratio,20.363043,22.708696,108.50
debt_to_gdp,8.500000,60.450000,209.40



Saved: C:\Users\Gebruiker\OneDrive - UniversitÃƒÆ’Ã‚Â  Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\MEP\7 Country level variables\demography_country_year.csv


In [ ]:
if "country_year_panel" in globals():
    assert len(country_year_panel) > 0, "country_year_panel is empty"

if "demography_country_year" in globals():
    assert len(demography_country_year) > 0, "demography_country_year is empty"
    if {"country", "year"}.issubset(demography_country_year.columns):
        dup_count = int(demography_country_year.duplicated(["country", "year"]).sum())
        assert dup_count == 0, f"Duplicate country-year rows found: {dup_count}"

if COUNTRY_PANEL_CSV_PATH.exists():
    print(f"Validated export path: {COUNTRY_PANEL_CSV_PATH}")